In [ ]:
BANNER = \
"""
# ----------------------------------------------------------------------
# PTS | Paulo Trigo Silva
# mLDm | MoP
# v01
# ----------------------------------------------------------------------
"""

In [ ]:
# ------------------------------------------------------------
# container runtime
# ------------------------------------------------------------

# note:
# this notebook assumes that notebook 01 completed successfully
# using the container runtime configured in 00_penguins_db_boot.py
#
# recall that 00_penguins_db_boot.py is your working copy of
# _mLDm_00_penguins_db_boot.py
#
# exactly one of these alternatives must be active:
#
# RUNTIME = "podman"
# # RUNTIME = "docker"
#
# or:
#
# # RUNTIME = "podman"
# RUNTIME = "docker"

In [ ]:
# ------------------------------------------------------------
# connect DuckDB to the integrated PostgreSQL data source
# ------------------------------------------------------------

import duckdb

# create an in-memory DuckDB connection
db = duckdb.connect()

# install and load the DuckDB PostgreSQL extension
# the INSTALL statement is required only the first time the extension is used
db.sql("""
INSTALL postgres;
LOAD postgres;
""")

# attach the PostgreSQL database to DuckDB
# the database will be accessible through the name "pg"
db.sql("""
ATTACH
'host=localhost port=5432 dbname=mldm user=mldm password=mldm'
AS pg
(TYPE postgres);
""")

print("PTS | DuckDB connected to PostgreSQL")

In [ ]:
# ------------------------------------------------------------
# inspect observations with missing island information
# ------------------------------------------------------------

# check whether geographic coordinates are available when island is missing
db.sql("""
SELECT
    id_penguin,
    date_time,
    latitude,
    longitude
FROM pg.public.observation
WHERE name_island IS NULL
ORDER BY id_penguin;
""").show()

# expected result:
# 20 observations

In [ ]:
# ------------------------------------------------------------
# calculate distances from observations to island references
#
# note:
# create the view in PostgreSQL so that the provided
# haversine_km function can be used directly
# ------------------------------------------------------------

db.sql("""
CALL postgres_execute(
    'pg',
    '
    CREATE OR REPLACE VIEW observation_island_distances AS
    SELECT
        o.id_penguin,
        o.date_time,
        i.name AS island,
        haversine_km(
            o.latitude,
            o.longitude,
            i.centroid_latitude,
            i.centroid_longitude
        ) AS distance_km
    FROM observation AS o
    CROSS JOIN island AS i
    WHERE o.name_island IS NULL
    '
);
""")

# inspect distances from observations to islands
db.sql("""
SELECT *
FROM pg.public.observation_island_distances
ORDER BY id_penguin, date_time, distance_km;
""").show()

# expected result:
# 60 distances = 20 observations × 3 island references

In [ ]:
# ------------------------------------------------------------
# obtain the minimum island distance for each observation
# ------------------------------------------------------------

db.sql("""
CREATE OR REPLACE TEMP VIEW observation_min_distance AS
SELECT id_penguin,
       date_time,
       -- TODO: obtain the minimum distance for each observation
       ... AS distance_km
FROM pg.public.observation_island_distances

-- TODO: group distances by the attributes that identify an observation
GROUP BY
    ...;
""")

# inspect minimum distances from observations to islands
db.sql("""
SELECT *
FROM observation_min_distance
ORDER BY distance_km;
""").show()

# expected result:
# 20 rows — one minimum distance per observation

In [ ]:
# ------------------------------------------------------------
# identify the nearest island for each observation
# ------------------------------------------------------------

db.sql("""
CREATE OR REPLACE TEMP VIEW observation_nearest_island AS
SELECT d.id_penguin,
       d.date_time,
       d.island,
       d.distance_km
FROM pg.public.observation_island_distances AS d JOIN observation_min_distance AS m
     -- TODO: match each candidate distance with the corresponding observation
     ON ...

     -- TODO: retain the candidate whose distance is the minimum
     AND ...;
""")

db.sql("""
SELECT *
FROM observation_nearest_island
ORDER BY id_penguin;
""").show()

# expected result:
# 20 rows — one nearest island per observation

In [ ]:
# ------------------------------------------------------------
# update observations with the inferred island
# ------------------------------------------------------------

# assign the nearest island to observations with missing island information
db.sql("""
UPDATE pg.public.observation AS o

       -- TODO: assign the inferred island to the missing island attribute
       SET name_island = ...

FROM observation_nearest_island AS n
WHERE o.id_penguin = n.id_penguin
  AND o.date_time = n.date_time
  AND o.name_island IS NULL;
""")

# verify that all observations now have an island
db.sql("""
SELECT COUNT(*) AS n_missing_island
FROM pg.public.observation
WHERE name_island IS NULL;
""").show()

# expected result:
# n_missing_island = 0

In [ ]:
# ------------------------------------------------------------
# remove the auxiliary view used during source completion
# ------------------------------------------------------------

# remove the postgresql view so the database is left without completion-specific objects
db.sql("""
CALL postgres_execute(
    'pg',
    'DROP VIEW IF EXISTS observation_island_distances'
);
""")

# Visual checkpoint with Orange Data Mining (DM)

*Note:* Before completing the **final evidence question** (cf. next cell), take a first visual look at the collected data using **Orange DM**.

## 1. Install Orange DM

Orange DM is used as a **standalone application** and should **NOT** be installed in the course Python virtual environment (`__venv`).

If Orange DM is not already installed:

**1.a. Open the Orange DM download page**

`https://orangedatamining.com/download/`

**1.b. Select the installer for your operating system**

- **Windows** — download and run the Windows installer
- **macOS (Apple silicon)** — use the Apple silicon version
- **macOS (Intel)** — use the Intel version

**1.c. Complete the installation**

Start **Orange Data Mining** after the installation finishes.

## 2. Small visual exploration

**2.a. Load the collected data**

Add a `File` widget and load:

`01_input/_mLDm_00_penguins_data_collected.csv`

**2.b. Select the attributes**

Add a `Select Columns` widget and connect `File` to `Select Columns`.

In `Select Columns`:

- keep `species` as the **Target**
- choose a subset of attributes to keep as **Features**
- move attributes that you do not want to use directly in this first analysis to **Meta Attributes** or **Ignored**

Do not try to find the “best” feature set at this stage.

The goal is to notice that an attribute may exist in the source but still be excluded from, or assigned a different role in, an analytical view of the data.

## 3. Explore the data visually

Connect the output of `Select Columns` to:

- `Data Table`
- `Distributions`
- `Scatter Plot`

Then:

- use `Data Table` to inspect the resulting data
- use `Distributions` to inspect at least one numerical attribute
- use `Scatter Plot` to explore the relationship between two numerical attributes, using `species` to distinguish the penguins

## 4. Evidence

Save the Orange workflow as:

`02_code/penguins_visual_check.ows`

Save evidence of the Orange exploration in `04_report`.

The evidence must show:

- the workflow and widgets used
- the role assigned to `species`
- the selected and excluded attributes
- at least one visual result

You may use two screenshots, for example:

`04_report/penguins_orange_columns.png`

`04_report/penguins_orange_visualisation.png`

In [ ]:
# ------------------------------------------------------------
# TODO
# final evidence question
# ------------------------------------------------------------

# using evidence from the execution of notebooks 01 and 02, explain:
#
# - why the number of observations differs from the number of distinct penguins
# - how the completion process inferred the missing island information
# - what evidence confirms that the source was completed without changing its
#   observation-level structure
#
# answer in approximately 4–6 sentences
#
# A: